# 04 — Out-of-Sample Volatility Forecasting

## Objective

This notebook tests whether the multi-timescale volatility structure identified in the preceding analysis provides genuine out-of-sample predictive value.

The central comparison is between:

$$
\text{Single OU}
\qquad\text{and}\qquad
\text{Multi-OU}.
$$

Both models use the same realised-variance observations, intraday seasonal adjustment, state-space filtering framework and market-time convention. Their principal structural difference is the number of volatility-memory timescales.

For an $n$-factor model, the latent state satisfies

$$
X_t
=
\sum_{i=1}^{n}X_t^{(i)},
$$

and the filtered state distribution is

$$
X_t \mid \mathcal F_t
\sim
N(m_t,P_t).
$$

Because the OU transition distribution is known exactly, the latent state can be propagated to arbitrary future horizons without time-discretisation error.

Closed-market evolution is represented through an effective volatility clock,

$$
d\tau
=
\begin{cases}
dt, & \text{market open},\\[4pt]
c_{\mathrm{night}}\,dt, & \text{market closed}.
\end{cases}
$$

The parameter $c_{\mathrm{night}}$ is selected exclusively on the 2017–2018 validation sample.

Forecasts are evaluated for average future daily realised variance over

$$
h\in\{1,7,30,90\}
$$

trading-day horizons,

$$
\overline{RV}_{t,h}
=
\frac{1}{h}
\sum_{k=1}^{h}RV_{t+k}.
$$

The forecasting experiment therefore asks:

> **Does explicitly modelling multiple volatility-memory timescales improve out-of-sample realised-variance forecasts relative to a single-timescale OU benchmark?**

The final post-2019 test sample is used only once model parameters and the overnight clock have been frozen.

In [1]:
import sys
import json

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.append("..")

from implementation.market_clock import MarketClock
from implementation.multi_ou import MultiOU
from implementation.multi_ou_filter import MultiOUFilter
from implementation.volatility_forecaster import VolatilityForecaster
from implementation.volatility_backtest import backtest_model

%load_ext autoreload
%autoreload 2

In [2]:
rv_data = pd.read_csv("../data/SPY_10min_rv.csv")

rv_data["interval_start"] = pd.to_datetime(rv_data["interval_start"])
rv_data["interval_end"] = pd.to_datetime(rv_data["interval_end"])

rv_data = rv_data[
    rv_data["valid"]
    & rv_data["log_realised_variance"].notna()
].copy()

rv_data = rv_data.sort_values("interval_start").reset_index(drop=True)

seasonal_mean = pd.read_csv(
    "../data/training_seasonal_profile.csv",
    index_col=0
).squeeze("columns")

with open("../data/ou_parameters.json", "r") as file:
    parameters = json.load(file)

rv_data["clock_time"] = rv_data["interval_start"].dt.strftime("%H:%M")
rv_data["seasonal_mean"] = rv_data["clock_time"].map(seasonal_mean)
rv_data["adjusted_log_rv"] = rv_data["log_realised_variance"] - rv_data["seasonal_mean"]

In [3]:
train_end = pd.Timestamp("2017-01-01")
validation_end = pd.Timestamp("2019-01-01")

train = rv_data[rv_data["interval_start"] < train_end].copy()

validation = rv_data[
    (rv_data["interval_start"] >= train_end)
    & (rv_data["interval_start"] < validation_end)
].copy()

test = rv_data[
    rv_data["interval_start"] >= validation_end
].copy()

forecast_horizons = [1, 7, 30, 90]
candidate_clocks = [0, 0.25, 0.5, 0.75, 1]

In [4]:
minute_data = pd.read_csv("../data/SPY_clean_1min.csv")

minute_data["datetime"] = pd.to_datetime(minute_data["datetime"])
minute_data = minute_data.sort_values("datetime").reset_index(drop=True)

minute_data["date"] = minute_data["datetime"].dt.normalize()
minute_data["previous_close"] = minute_data.groupby("date")["Close"].shift(1)

minute_data["log_return"] = np.log(
    minute_data["Close"] / minute_data["previous_close"]
)

minute_data["squared_return"] = minute_data["log_return"] ** 2

daily_rv = minute_data.groupby("date").agg(
    realised_variance=("squared_return", "sum"),
    observations=("Close", "count")
).reset_index()

for horizon in forecast_horizons:
    future_rv = daily_rv["realised_variance"].shift(-1)

    daily_rv[f"target_{horizon}d"] = (
        future_rv
        .rolling(horizon, min_periods=horizon)
        .mean()
        .shift(-(horizon - 1))
    )

In [5]:
single_process = MultiOU(
    mean=0,
    mean_reversions=[parameters["single_ou"]["mean_reversion"]],
    weights=[parameters["single_ou"]["weight"]]
)

multi_process = MultiOU(
    mean=0,
    mean_reversions=parameters["multi_ou"]["mean_reversions"],
    weights=parameters["multi_ou"]["weights"]
)

In [6]:
validation_results = backtest_model(
    process=multi_process,
    observation_variance=parameters["multi_ou"]["observation_variance"],
    rv_data=rv_data,
    daily_rv=daily_rv,
    seasonal_mean=seasonal_mean,
    c_night=0.5,
    start_date=train_end,
    end_date=validation_end,
    horizons=forecast_horizons
)

In [7]:
validation_results.head()

,date,c_night,forecast_1d,actual_1d,forecast_7d,actual_7d,forecast_30d,actual_30d,forecast_90d,actual_90d
0,2017-01-03,0.5,0.000023,0.000011,0.000028,0.000017,0.000044,0.000013,0.000063,0.000014
1,2017-01-04,0.5,0.000017,0.000013,0.000021,0.000017,0.000038,0.000013,0.000060,0.000014
2,2017-01-05,0.5,0.000014,0.000014,0.000019,0.000017,0.000035,0.000013,0.000058,0.000014
3,2017-01-06,0.5,0.000017,0.000010,0.000021,0.000017,0.000036,0.000013,0.000059,0.000014
4,2017-01-09,0.5,0.000014,0.000010,0.000018,0.000017,0.000034,0.000013,0.000058,0.000015


In [8]:
print("Forecast origins:", len(validation_results))

for horizon in forecast_horizons:
    forecast = validation_results[f"forecast_{horizon}d"]
    actual = validation_results[f"actual_{horizon}d"]

    print()
    print(horizon, "day")
    print("Mean forecast:", forecast.mean())
    print("Mean actual:", actual.mean())
    print("Forecast / actual:", forecast.mean() / actual.mean())

Forecast origins: 502

1 day
Mean forecast: 4.3424463931415504e-05
Mean actual: 4.2278164044483805e-05
Forecast / actual: 1.0271132844303645

7 day
Mean forecast: 4.5164845889464915e-05
Mean actual: 4.302792713316535e-05
Forecast / actual: 1.0496635301460397

30 day
Mean forecast: 5.315632084749162e-05
Mean actual: 4.386578735464061e-05
Forecast / actual: 1.2117945226365157

90 day
Mean forecast: 6.673995195545847e-05
Mean actual: 4.480168246026871e-05
Forecast / actual: 1.4896751258090628


In [10]:
def qlike(actual, forecast):
    return actual / forecast + np.log(forecast)


def log_mse(actual, forecast):
    return np.mean((np.log(actual) - np.log(forecast)) ** 2)

validation_scores = []

for horizon in forecast_horizons:
    forecast = validation_results[f"forecast_{horizon}d"]
    actual = validation_results[f"actual_{horizon}d"]

    valid = forecast.notna() & actual.notna()

    forecast = forecast[valid]
    actual = actual[valid]

    validation_scores.append({
        "horizon": horizon,
        "qlike": qlike(actual, forecast).mean(),
        "log_mse": log_mse(actual, forecast),
        "observations": len(actual)
    })

validation_scores = pd.DataFrame(validation_scores)

validation_scores

,horizon,qlike,log_mse,observations
0,1,-9.675129,0.338186,502
1,7,-9.506353,0.408479,502
2,30,-9.206832,0.840338,502
3,90,-8.965974,1.180813,502


In [12]:
plot_data = validation_results[
    ["date", "forecast_1d", "actual_1d"]
].melt(
    id_vars="date",
    var_name="series",
    value_name="realised_variance"
)

fig = px.line(
    plot_data,
    x="date",
    y="realised_variance",
    color="series",
    title="1-Day Realised Variance Forecast — Validation"
)

fig.show()

In [14]:
plot_data = validation_results[
    ["date", "forecast_30d", "actual_30d"]
].melt(
    id_vars="date",
    var_name="series",
    value_name="realised_variance"
)

fig = px.line(
    plot_data,
    x="date",
    y="realised_variance",
    color="series",
    title="30-Day Mean Realised Variance Forecast — Validation"
)

fig.show()

In [ ]:
candidate_clocks = [0, 0.25, 0.5, 0.75, 1]

clock_results = []

for c_night in candidate_clocks:
    results = backtest_model(
        process=multi_process,
        observation_variance=parameters["multi_ou"]["observation_variance"],
        rv_data=rv_data,
        daily_rv=daily_rv,
        seasonal_mean=seasonal_mean,
        c_night=c_night,
        start_date=train_end,
        end_date=validation_end,
        horizons=forecast_horizons
    )

    for horizon in forecast_horizons:
        forecast = results[f"forecast_{horizon}d"]
        actual = results[f"actual_{horizon}d"]

        valid = forecast.notna() & actual.notna() & (forecast > 0) & (actual > 0)

        forecast = forecast[valid]
        actual = actual[valid]

        clock_results.append({
            "c_night": c_night,
            "horizon": horizon,
            "qlike": qlike(actual, forecast).mean(),
            "log_mse": log_mse(actual, forecast),
            "mean_ratio": forecast.mean() / actual.mean(),
            "observations": len(actual)
        })

clock_results = pd.DataFrame(clock_results)

In [18]:
clock_results

,c_night,horizon,qlike,log_mse,mean_ratio,observations
0,0.00,1,-9.667992,0.308962,1.027271,502
1,0.00,7,-9.485943,0.360327,0.968716,502
2,0.00,30,-9.145604,0.577985,1.006739,502
3,0.00,90,-8.956777,0.835905,1.148532,502
4,0.25,1,-9.674477,0.323832,1.012793,502
5,0.25,7,-9.502673,0.376521,1.015820,502
6,0.25,30,-9.219312,0.669203,1.118946,502
7,0.25,90,-8.992454,1.015020,1.363555,502
8,0.50,1,-9.675129,0.338186,1.027113,502
9,0.50,7,-9.506353,0.408479,1.049664,502


In [19]:
clock_selection = clock_results.copy()

clock_selection["qlike_excess"] = (
    clock_selection["qlike"]
    - clock_selection.groupby("horizon")["qlike"].transform("min")
)

clock_summary = (
    clock_selection
    .groupby("c_night")
    .agg(
        mean_excess_qlike=("qlike_excess", "mean"),
        mean_log_mse=("log_mse", "mean"),
        mean_ratio=("mean_ratio", "mean")
    )
    .reset_index()
)

clock_summary

,c_night,mean_excess_qlike,mean_log_mse,mean_ratio
0,0.00,0.034233,0.520795,1.037815
1,0.25,0.001083,0.596144,1.127779
2,0.50,0.009740,0.691954,1.194562
3,0.75,0.025218,0.778531,1.243405
4,1.00,0.041275,0.854299,1.281416


In [20]:
best_clock = clock_summary.loc[
    clock_summary["mean_excess_qlike"].idxmin(),
    "c_night"
]

print("Selected c_night:", best_clock)

Selected c_night: 0.25


In [21]:
single_clock_results = []

for c_night in candidate_clocks:
    results = backtest_model(
        process=single_process,
        observation_variance=parameters["single_ou"]["observation_variance"],
        rv_data=rv_data,
        daily_rv=daily_rv,
        seasonal_mean=seasonal_mean,
        c_night=c_night,
        start_date=train_end,
        end_date=validation_end,
        horizons=forecast_horizons
    )

    for horizon in forecast_horizons:
        forecast = results[f"forecast_{horizon}d"]
        actual = results[f"actual_{horizon}d"]

        valid = forecast.notna() & actual.notna() & (forecast > 0) & (actual > 0)
        forecast = forecast[valid]
        actual = actual[valid]

        single_clock_results.append({
            "c_night": c_night,
            "horizon": horizon,
            "qlike": qlike(actual, forecast).mean(),
            "log_mse": log_mse(actual, forecast),
            "mean_ratio": forecast.mean() / actual.mean(),
            "observations": len(actual)
        })

single_clock_results = pd.DataFrame(single_clock_results)

In [22]:
single_clock_results["qlike_excess"] = (
    single_clock_results["qlike"]
    - single_clock_results.groupby("horizon")["qlike"].transform("min")
)

single_clock_summary = (
    single_clock_results
    .groupby("c_night")
    .agg(
        mean_excess_qlike=("qlike_excess", "mean"),
        mean_log_mse=("log_mse", "mean"),
        mean_ratio=("mean_ratio", "mean")
    )
    .reset_index()
)

single_clock_summary

,c_night,mean_excess_qlike,mean_log_mse,mean_ratio
0,0.00,0.028462,0.575625,1.130364
1,0.25,0.004621,0.673646,1.217553
2,0.50,0.018374,0.780611,1.276250
3,0.75,0.036223,0.872641,1.318866
4,1.00,0.053546,0.951236,1.351916


In [23]:
best_single_clock = single_clock_summary.loc[
    single_clock_summary["mean_excess_qlike"].idxmin(),
    "c_night"
]

print("Selected Single-OU c_night:", best_single_clock)

Selected Single-OU c_night: 0.25


In [24]:
selected_clock = 0.25

single_test = backtest_model(
    process=single_process,
    observation_variance=parameters["single_ou"]["observation_variance"],
    rv_data=rv_data,
    daily_rv=daily_rv,
    seasonal_mean=seasonal_mean,
    c_night=selected_clock,
    start_date=validation_end,
    end_date=rv_data["interval_start"].max() + pd.Timedelta(days=1),
    horizons=forecast_horizons
)

multi_test = backtest_model(
    process=multi_process,
    observation_variance=parameters["multi_ou"]["observation_variance"],
    rv_data=rv_data,
    daily_rv=daily_rv,
    seasonal_mean=seasonal_mean,
    c_night=selected_clock,
    start_date=validation_end,
    end_date=rv_data["interval_start"].max() + pd.Timedelta(days=1),
    horizons=forecast_horizons
)

In [25]:
test_scores = []

for model_name, results in [
    ("Single OU", single_test),
    ("Multi-OU", multi_test)
]:
    for horizon in forecast_horizons:
        forecast = results[f"forecast_{horizon}d"]
        actual = results[f"actual_{horizon}d"]

        valid = (
            forecast.notna()
            & actual.notna()
            & (forecast > 0)
            & (actual > 0)
        )

        forecast = forecast[valid]
        actual = actual[valid]

        test_scores.append({
            "model": model_name,
            "horizon": horizon,
            "qlike": qlike(actual, forecast).mean(),
            "log_mse": log_mse(actual, forecast),
            "mean_ratio": forecast.mean() / actual.mean(),
            "observations": len(actual)
        })

test_scores = pd.DataFrame(test_scores)

test_scores

,model,horizon,qlike,log_mse,mean_ratio,observations
0,Single OU,1,-8.925929,0.387264,0.978482,1840
1,Single OU,7,-8.770554,0.395768,0.953633,1840
2,Single OU,30,-8.545777,0.486722,0.941454,1840
3,Single OU,90,-8.443143,0.535794,0.978046,1840
4,Multi-OU,1,-8.952885,0.360333,0.942129,1840
5,Multi-OU,7,-8.800763,0.345822,0.927110,1840
6,Multi-OU,30,-8.538015,0.459482,0.921610,1840
7,Multi-OU,90,-8.416845,0.525001,0.957652,1840


In [26]:
score_comparison = test_scores.pivot(
    index="horizon",
    columns="model",
    values="qlike"
)

score_comparison["qlike_improvement"] = (
    (score_comparison["Single OU"] - score_comparison["Multi-OU"])
    / np.abs(score_comparison["Single OU"])
)

score_comparison

model,Multi-OU,Single OU,qlike_improvement
horizon,,,
1,-8.952885,-8.925929,0.003020
7,-8.800763,-8.770554,0.003444
30,-8.538015,-8.545777,-0.000908
90,-8.416845,-8.443143,-0.003115


In [27]:
score_comparison["qlike_difference"] = (
    score_comparison["Single OU"]
    - score_comparison["Multi-OU"]
)

score_comparison

model,Multi-OU,Single OU,qlike_improvement,qlike_difference
horizon,,,,
1,-8.952885,-8.925929,0.003020,0.026957
7,-8.800763,-8.770554,0.003444,0.030209
30,-8.538015,-8.545777,-0.000908,-0.007762
90,-8.416845,-8.443143,-0.003115,-0.026298


In [28]:
log_mse_comparison = test_scores.pivot(
    index="horizon",
    columns="model",
    values="log_mse"
)

log_mse_comparison["improvement"] = (
    (log_mse_comparison["Single OU"] - log_mse_comparison["Multi-OU"])
    / log_mse_comparison["Single OU"]
)

log_mse_comparison

model,Multi-OU,Single OU,improvement
horizon,,,
1,0.360333,0.387264,0.069542
7,0.345822,0.395768,0.126201
30,0.459482,0.486722,0.055966
90,0.525001,0.535794,0.020145


In [29]:
bias_comparison = test_scores.pivot(
    index="horizon",
    columns="model",
    values="mean_ratio"
)

bias_comparison

model,Multi-OU,Single OU
horizon,,
1,0.942129,0.978482
7,0.927110,0.953633
30,0.921610,0.941454
90,0.957652,0.978046


In [36]:
single_scores = test_scores[
    test_scores["model"] == "Single OU"
].set_index("horizon")

multi_scores = test_scores[
    test_scores["model"] == "Multi-OU"
].set_index("horizon")

comparison = pd.DataFrame({
    "single_ou_log_mse": single_scores["log_mse"],
    "multi_ou_log_mse": multi_scores["log_mse"],
    "log_mse_improvement": (
        single_scores["log_mse"] - multi_scores["log_mse"]
    ) / single_scores["log_mse"],
    "single_ou_qlike": single_scores["qlike"],
    "multi_ou_qlike": multi_scores["qlike"],
    "qlike_difference": (
        single_scores["qlike"] - multi_scores["qlike"]
    )
})

comparison

,single_ou_log_mse,multi_ou_log_mse,log_mse_improvement,single_ou_qlike,multi_ou_qlike,qlike_difference
horizon,,,,,,
1,0.387264,0.360333,0.069542,-8.925929,-8.952885,0.026957
7,0.395768,0.345822,0.126201,-8.770554,-8.800763,0.030209
30,0.486722,0.459482,0.055966,-8.545777,-8.538015,-0.007762
90,0.535794,0.525001,0.020145,-8.443143,-8.416845,-0.026298


In [30]:
log_mse_plot = log_mse_comparison.reset_index()

fig = px.bar(
    log_mse_plot,
    x="horizon",
    y="improvement",
    title="Out-of-Sample Log-MSE Improvement: Multi-OU vs Single OU"
)

fig.update_layout(
    xaxis_title="Forecast Horizon (Trading Days)",
    yaxis_title="Log-MSE Improvement"
)

fig.show()

In [34]:
qlike_data = comparison.reset_index()

fig = px.bar(
    qlike_data,
    x="horizon",
    y="qlike_difference",
    title="Out-of-Sample QLIKE Difference: Single OU − Multi-OU"
)

fig.update_layout(
    xaxis_title="Forecast Horizon (Trading Days)",
    yaxis_title="QLIKE Difference"
)

fig.add_hline(y=0)

fig.show()

In [35]:
forecast_comparison = pd.DataFrame({
    "date": multi_test["date"],
    "actual": multi_test["actual_7d"],
    "multi_ou": multi_test["forecast_7d"],
    "single_ou": single_test["forecast_7d"]
})

forecast_comparison = forecast_comparison.melt(
    id_vars="date",
    var_name="series",
    value_name="realised_variance"
)

fig = px.line(
    forecast_comparison,
    x="date",
    y="realised_variance",
    color="series",
    title="7-Day Realised Variance Forecasts — Out-of-Sample"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Mean Daily Realised Variance"
)

fig.show()

## Conclusion

The out-of-sample forecasting experiment provides evidence that modelling multiple volatility-memory timescales improves realised-variance forecasting, although the magnitude of the improvement depends on both forecast horizon and loss function.

The overnight market clock was selected using the 2017–2018 validation period. Both Single-OU and Multi-OU models independently selected

$$
\boxed{c_{\mathrm{night}}=0.25},
$$

indicating that the fitted latent volatility process continues to evolve while the market is closed, but at a slower rate than during trading hours.

After this selection, all model parameters were frozen and the specifications were evaluated on the untouched post-2019 test sample.

Relative to Single OU, Multi-OU reduced out-of-sample log-forecast MSE by:

| Forecast horizon | Log-MSE reduction |
| ---: | ---: |
| 1 trading day | 7.0% |
| 7 trading days | 12.6% |
| 30 trading days | 5.6% |
| 90 trading days | 2.0% |

The largest improvement occurs at the 7-day horizon.

This horizon dependence is consistent with the structure of the model. At very short horizons, the current volatility level dominates and a single persistence factor can approximate the dynamics reasonably well. At intermediate horizons, separating a rapidly mean-reverting component from a persistent volatility regime becomes particularly valuable:

$$
E_t[X_{t+\tau}]
=
X_t^{(f)}e^{-\lambda_f\tau}
+
X_t^{(s)}e^{-\lambda_s\tau}.
$$

At sufficiently long horizons, both components increasingly revert towards their stationary means, reducing the incremental forecasting value of the additional factor.

QLIKE produces a more nuanced result. Multi-OU outperforms Single OU at the 1- and 7-day horizons, while Single OU performs marginally better at the 30- and 90-day horizons. The Multi-OU model therefore does not uniformly dominate under every scoring rule.

Taken together, the results support three conclusions:

1. SPY volatility exhibits robust dependence across multiple relaxation timescales.
2. A parsimonious two-factor OU model captures this empirical memory structure substantially better than a single exponential relaxation model.
3. The additional structure translates into measurable out-of-sample predictive value, with the strongest improvement at short-to-medium forecasting horizons.

The results also highlight an important limitation: fine features of the long-memory spectrum are weakly identified, and improved structural fit does not guarantee uniform forecasting dominance at every horizon or under every loss function.

Overall, the project demonstrates how empirical volatility memory can be translated from a non-parametric covariance structure into a continuous-time latent-factor model, filtered sequentially and evaluated under a leakage-controlled out-of-sample forecasting design.